# TEST-06: Drawdown Semantics

**WP-5a Pipeline Audit** | 2026-04-16

**Intent:** Determine the max-drawdown formula and whether it can exceed 100%.

**Formula under investigation** (`lib/evaluation.py`, `max_drawdown()`):
```python
peak = equity_curve.cummax()
drawdown = (equity_curve - peak) / peak
max_dd = float(-drawdown.min())
```

**Key question:** If `equity_curve` dips below zero (no liquidation modeled), `drawdown = (negative - peak) / peak` can produce values < -1, making `max_dd > 1.0` (>100%).

**Context:** trial_log entries show max_dd values of 1.78, 1.21, 1.10, 2.12, 6.12 (178%, 120%, 110%, 212%, 612% respectively) for H-001 through H-006. These exceed 100%, indicating equity went below zero in each case.

In [1]:
import sys
from pathlib import Path
import pandas as pd
sys.path.insert(0, 'C:/GitHub/crypto-perp-wedge')

from lib.evaluation import max_drawdown

print('=== Formula documentation ===')
print('peak = equity_curve.cummax()')
print('drawdown = (equity_curve - peak) / peak')
print('max_dd = float(-drawdown.min())')
print()
print('Key property: if equity < 0 and peak > 0, then:')
print('  drawdown = (equity - peak) / peak = equity/peak - 1')
print('  If equity = -2k, peak = 10k: drawdown = -2000/10000 - 1 = -1.2')
print('  -> max_dd = 1.2 = 120% (exceeds 100%)')

=== Formula documentation ===
peak = equity_curve.cummax()
drawdown = (equity_curve - peak) / peak
max_dd = float(-drawdown.min())

Key property: if equity < 0 and peak > 0, then:
  drawdown = (equity - peak) / peak = equity/peak - 1
  If equity = -2k, peak = 10k: drawdown = -2000/10000 - 1 = -1.2
  -> max_dd = 1.2 = 120% (exceeds 100%)


In [2]:
# Case A: bounded equity curve
eq_a = pd.Series(
    [10_000.0, 11_000.0, 12_000.0, 10_000.0, 6_000.0, 1_000.0, 1_500.0, 2_000.0],
    index=pd.date_range('2020-01-01', periods=8, freq='4h', tz='UTC')
)
dd_a, peak_a, trough_a = max_drawdown(eq_a)
expected_dd_a = (12_000 - 1_000) / 12_000

print('=== Case A: Bounded equity curve (equity always >= 0) ===')
print(f'  Equity curve: {list(eq_a.values)}')
print(f'  Peak:   12,000  at {peak_a}')
print(f'  Trough:  1,000  at {trough_a}')
print(f'  Expected DD: (12000-1000)/12000 = {expected_dd_a:.6f} = {expected_dd_a*100:.4f}%')
print(f'  Actual DD:   {dd_a:.6f} = {dd_a*100:.4f}%')
print(f'  Match: {abs(dd_a - expected_dd_a) < 1e-6}')
print(f'  Exceeds 100%: {dd_a > 1.0} (expected: False)')

=== Case A: Bounded equity curve (equity always >= 0) ===
  Equity curve: [10000.0, 11000.0, 12000.0, 10000.0, 6000.0, 1000.0, 1500.0, 2000.0]
  Peak:   12,000  at 2020-01-01 08:00:00+00:00
  Trough:  1,000  at 2020-01-01 20:00:00+00:00
  Expected DD: (12000-1000)/12000 = 0.916667 = 91.6667%
  Actual DD:   0.916667 = 91.6667%
  Match: True
  Exceeds 100%: False (expected: False)


In [3]:
# Case B: equity goes below zero (no liquidation)
eq_b = pd.Series(
    [10_000.0, 11_000.0, 12_000.0, 5_000.0, -10_000.0, -40_000.0],
    index=pd.date_range('2020-01-01', periods=6, freq='4h', tz='UTC')
)
dd_b, peak_b, trough_b = max_drawdown(eq_b)
expected_dd_b = (12_000 + 40_000) / 12_000  # (peak - trough) / peak = (12k - (-40k)) / 12k

print('=== Case B: Equity goes negative (no liquidation modeled) ===')
print(f'  Equity curve: {list(eq_b.values)}')
print(f'  Peak:   12,000  at {peak_b}')
print(f'  Trough: -40,000 at {trough_b}')
print(f'  Formula: (-40000 - 12000) / 12000 = {(-40000-12000)/12000:.4f}')
print(f'  max_dd = -{(-40000-12000)/12000:.4f} = {expected_dd_b:.4f} = {expected_dd_b*100:.2f}%')
print(f'  Expected DD: {expected_dd_b:.6f} = {expected_dd_b*100:.4f}%')
print(f'  Actual DD:   {dd_b:.6f} = {dd_b*100:.4f}%')
print(f'  Match: {abs(dd_b - expected_dd_b) < 1e-6}')
print(f'  Exceeds 100%: {dd_b > 1.0} <-- CONFIRMED')

=== Case B: Equity goes negative (no liquidation modeled) ===
  Equity curve: [10000.0, 11000.0, 12000.0, 5000.0, -10000.0, -40000.0]
  Peak:   12,000  at 2020-01-01 08:00:00+00:00
  Trough: -40,000 at 2020-01-01 20:00:00+00:00
  Formula: (-40000 - 12000) / 12000 = -4.3333
  max_dd = -(-4.3333) = 4.3333 = 433.33%
  Expected DD: 4.333333 = 433.3333%
  Actual DD:   4.333333 = 433.3333%
  Match: True
  Exceeds 100%: True <-- CONFIRMED


In [4]:
print('=== Existing trial_log evidence ===')
trial_log_evidence = [
    ('H-001', 1.7813, '178.1%', 'equity went below zero'),
    ('H-002', 1.2050, '120.5%', 'equity went below zero'),
    ('H-003', 1.0959, '109.6%', 'equity went below zero'),
    ('H-004', 0.6979, '69.8%', 'bounded (below 100%)'),
    ('H-005', 2.1208, '212.1%', 'final equity=-35,556 from 10,000 initial'),
    ('H-006', 6.1208, '612.1%', 'equity went deeply negative'),
    ('H-007', 1.1523, '115.2%', 'equity went below zero'),
    ('H-010', 0.8497, '84.97%', 'bounded (below 100%)'),
]
for h, dd, pct, note in trial_log_evidence:
    print(f'  {h}: max_dd={dd:.4f} ({pct}) -- {note}')
print()
print('=== VERDICT: MATERIAL FINDING ===')
print('The max_drawdown formula produces values > 1.0 whenever equity goes below zero.')
print('The pipeline does NOT model liquidation or margin calls.')
print('Equity can become arbitrarily negative, making max_dd unbounded.')
print()
print('Implication for existing verdicts:')
print('  All 8 executed hypotheses received NO-GO primarily on Sharpe < 1.0.')
print('  The DD metric is ADDITIONAL evidence of failure, not the primary gate.')
print('  NO-GO verdicts remain valid -- DD > 100% means the hypothesis would have')
print('  been liquidated in practice, which only strengthens the rejection.')
print()
print('Fix required in a future package:')
print('  Option A: cap equity at 0 (model liquidation) -- changes backtest semantics')
print('  Option B: report DD as min(max_dd, 1.0) with a "liquidation_triggered" flag')
print('  Option C: add documentation noting DD > 100% means unrealistic scenario')

=== Existing trial_log evidence ===
  H-001: max_dd=1.7813 (178.1%) -- equity went below zero
  H-002: max_dd=1.2050 (120.5%) -- equity went below zero
  H-003: max_dd=1.0959 (109.6%) -- equity went below zero
  H-004: max_dd=0.6979 (69.8%) -- bounded (below 100%)
  H-005: max_dd=2.1208 (212.1%) -- final equity=-35,556 from 10,000 initial
  H-006: max_dd=6.1208 (612.1%) -- equity went deeply negative
  H-007: max_dd=1.1523 (115.2%) -- equity went below zero
  H-010: max_dd=0.8497 (84.97%) -- bounded (below 100%)

=== VERDICT: MATERIAL FINDING ===
The max_drawdown formula produces values > 1.0 whenever equity goes below zero.
The pipeline does NOT model liquidation or margin calls.
Equity can become arbitrarily negative, making max_dd unbounded.

Implication for existing verdicts:
  All 8 executed hypotheses received NO-GO primarily on Sharpe < 1.0.
  The DD metric is ADDITIONAL evidence of failure, not the primary gate.
  NO-GO verdicts remain valid -- DD > 100% means the hypothesis wo